# Train π0.5 (`pi05`) on SO-101 for Camera-Robust Grasping

Fine-tune **LeRobot π0.5** (2B PaliGemma VLM backbone) on your SO-101 dataset with **full model fine-tuning** and **image augmentations** to maximize generalization to a shifted camera angle.

- **Dataset:** `AdithyaRajendran/so101_grab_brain_t2` (241 episodes)
- **Task:** *"Grab the grey brain toy and place it inside the green container"*
- **Robot:** `so101_follower`
- **Policy:** `pi05` (full fine-tune, vision encoder unfrozen)

### Strategy: Camera Angle Robustness
Since the physical camera has moved since data collection:
1. **Full model fine-tuning** (not expert-only) — lets the vision encoder adapt
2. **Image augmentations** — RandomAffine + ColorJitter + SharpnessJitter simulate viewpoint variation
3. **Descriptive task prompt** — helps VLM ground semantically rather than memorize pixel patterns

### Recommended runtime: **H100 80GB**
- Full fine-tune with batch_size=32, ~1.5-2.5 hrs for 5000 steps
- ~30-50 CU on Colab Pro

In [42]:

# Colab: check GPU
import torch, os, platform, subprocess, json

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))
    print("Compute capability:", f"{props.major}.{props.minor}")
else:
    raise RuntimeError("Please switch Colab runtime to GPU: Runtime > Change runtime type > GPU")


Python: 3.12.12
Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
VRAM (GB): 79.18
Compute capability: 9.0


In [43]:
# Set output directory
import os
OUTPUT_DIR = "/content/outputs/pi05_so101_grab_brain_t2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")

Outputs will be saved to: /content/outputs/pi05_so101_grab_brain_t2


In [44]:
%%bash
# Install LeRobot + π0.5 dependencies
cd /content

if [ ! -d lerobot ]; then
  git clone https://github.com/huggingface/lerobot.git
fi

cd /content/lerobot

apt-get -qq update
apt-get -qq install -y ffmpeg

# Downgrade Colab pre-installed packages that conflict with LeRobot 0.4.5's pinned ranges
pip install -q "huggingface-hub[cli,hf-transfer]>=0.34.2,<0.36.0" "setuptools>=71.0.0,<81.0.0" "wandb>=0.24.0,<0.25.0"

# Now install LeRobot — deps already satisfied, no conflicts
pip install -q -e ".[pi]"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



## Authenticate

You should log in to:
- **Hugging Face** so the training job can read your dataset and optionally push checkpoints
- **Weights & Biases** only if you want experiment tracking


In [45]:
%%bash
# Authenticate with Hugging Face
huggingface-cli login --token YOUR_HF_TOKEN_HERE

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Pi0.5` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Pi0.5`


In [46]:
# Enable W&B for experiment tracking
import wandb
USE_WANDB = True
wandb.login(key="YOUR_WANDB_KEY_HERE")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [47]:
# Training configuration — H100 optimized for camera angle robustness
import torch, os, json
from pathlib import Path

DATASET_REPO_ID = "AdithyaRajendran/so101_grab_brain_t2"
POLICY_REPO_ID = "AdithyaRajendran/pi05_so101_grab_brain_t2"
JOB_NAME = "pi05_so101_grab_brain_t2"

# H100 optimized settings
DTYPE = "bfloat16"
BATCH_SIZE = 32
TRAIN_EXPERT_ONLY = False     # CRITICAL: full fine-tuning for camera generalization
FREEZE_VISION_ENCODER = False  # CRITICAL: let vision encoder adapt to your scene
COMPILE_MODEL = True           # H100 supports torch.compile well, ~20% speedup
STEPS = 5000                   # More steps for full fine-tune (was 3000)

# Keep MEAN_STD fallback since dataset doesn't have quantile stats
NORMALIZATION_MAPPING = {
    "ACTION": "MEAN_STD",
    "STATE": "MEAN_STD",
    "VISUAL": "IDENTITY",
}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print({
    "DATASET_REPO_ID": DATASET_REPO_ID,
    "POLICY_REPO_ID": POLICY_REPO_ID,
    "OUTPUT_DIR": OUTPUT_DIR,
    "DTYPE": DTYPE,
    "BATCH_SIZE": BATCH_SIZE,
    "TRAIN_EXPERT_ONLY": TRAIN_EXPERT_ONLY,
    "FREEZE_VISION_ENCODER": FREEZE_VISION_ENCODER,
    "COMPILE_MODEL": COMPILE_MODEL,
    "STEPS": STEPS,
})

{'DATASET_REPO_ID': 'AdithyaRajendran/so101_grab_brain_t2', 'POLICY_REPO_ID': 'AdithyaRajendran/pi05_so101_grab_brain_t2', 'OUTPUT_DIR': '/content/outputs/pi05_so101_grab_brain_t2', 'DTYPE': 'bfloat16', 'BATCH_SIZE': 32, 'TRAIN_EXPERT_ONLY': False, 'FREEZE_VISION_ENCODER': False, 'COMPILE_MODEL': True, 'STEPS': 5000}



## Optional: add quantile stats to the dataset

The π0.5 docs say that if the dataset is **not converted with quantiles**, you can either:

1. run the quantile augmentation script, or  
2. train with a fallback normalization mapping.

This notebook already uses the fallback normalization mapping by default, so you do **not** need to run the next cell unless you explicitly want quantile stats in the dataset repo.


In [48]:

# OPTIONAL: only run this if you want to augment the dataset repo with quantile stats
# and you have write access to that dataset on the Hub.

RUN_QUANTILE_AUGMENT = False

if RUN_QUANTILE_AUGMENT:
    !python src/lerobot/datasets/v30/augment_dataset_quantile_stats.py \
        --repo-id=$DATASET_REPO_ID


In [51]:
%%bash
# Launch pi0.5 training with image augmentations for camera angle robustness
cd /content/lerobot

# Clean output dir from previous failed runs
rm -rf /content/outputs/pi05_so101_grab_brain_t2

# pi05_base expects: base_0_rgb, left_wrist_0_rgb, right_wrist_0_rgb
# Dataset has: front, wrist → remap + empty_cameras=1 for missing 3rd camera
# NOTE: batch_size=16, compile_model=false to fit in H100 80GB with full fine-tune

python src/lerobot/scripts/lerobot_train.py \
    --dataset.repo_id=AdithyaRajendran/so101_grab_brain_t2 \
    --dataset.image_transforms.enable=true \
    --output_dir=/content/outputs/pi05_so101_grab_brain_t2 \
    --job_name=pi05_so101_grab_brain_t2 \
    --policy.path=lerobot/pi05_base \
    --policy.compile_model=false \
    --policy.gradient_checkpointing=true \
    --policy.dtype=bfloat16 \
    --policy.freeze_vision_encoder=false \
    --policy.train_expert_only=false \
    --policy.repo_id=AdithyaRajendran/pi05_so101_grab_brain_t2 \
    --steps=5000 \
    --policy.device=cuda \
    --batch_size=16 \
    '--policy.normalization_mapping={"ACTION": "MEAN_STD", "STATE": "MEAN_STD", "VISUAL": "IDENTITY"}' \
    '--rename_map={"observation.images.front": "observation.images.base_0_rgb", "observation.images.wrist": "observation.images.left_wrist_0_rgb"}' \
    --policy.empty_cameras=1 \
    --wandb.enable=true \
    --save_freq=1000 \
    --log_freq=100

The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi
Loading model from: lerobot/pi05_base
✓ Loaded state dict from model.safetensors
Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.d

2026-03-02 11:40:59.089811: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 11:40:59.102676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772451659.119211   36503 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772451659.124406   36503 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772451659.137989   36503 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## After training — Checkpoint Selection

Checkpoints are saved every 1000 steps at:
`/content/outputs/pi05_so101_grab_brain_t2/`

### How to pick the best checkpoint
- **5 checkpoints** saved: steps 1000, 2000, 3000, 4000, 5000
- If training loss plateaus early, an earlier checkpoint may generalize better (less overfitting to training camera angle)
- **Deploy each checkpoint on the robot** at the new camera angle to find the sweet spot
- Start testing from step 3000 onwards — earlier checkpoints may be undertrained

### If it still doesn't work
1. Try `lerobot/pi05_libero` as base instead of `pi05_base`
2. Increase augmentation strength: `degrees=[-15,15]`, `translate=[0.15,0.15]`
3. Consider SARM+RA-BC as a second attempt
4. Even 10-15 new episodes at the new camera angle would be the strongest fix

In [ ]:

# Quick listing of saved outputs
import os
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files[:20]:
        print(f"{subindent}{f}")
    if level >= 2:
        dirs[:] = []


## Troubleshooting

- **OOM on H100**: Reduce `BATCH_SIZE` to 16, keep everything else the same
- **Loss spikes or NaN**: Reduce batch size to 16; if persists, try `COMPILE_MODEL = False`
- **Loss not decreasing**: Verify dataset is loading correctly; check WandB dashboard
- **Running on smaller GPU** (A100 40GB / L4):
  - Set `BATCH_SIZE = 8`, `TRAIN_EXPERT_ONLY = True`, `COMPILE_MODEL = False`
  - Keep `FREEZE_VISION_ENCODER = False` and augmentations enabled